<a href="https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Om-Ranmode/flyrank-ml-internshipPractice/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Archetype to Action Mapping
We convert raw model prediction scores and historical traffic indicators into four human-actionable content archetypes:

Immediate Refresh (REWRITE_NOW): High historical traffic (impressions_90d >= 500) experiencing significant predicted decline (pred == 1). Action: Refresh search intent, expand technical depth, update outdated facts.

Metadata & CTR Tune (CTR_OPTIMIZE): Page ranks on Page 1 or 2 (avg_position <= 15) but exhibits below-average click-through rates (ctr < 0.02). Action: Rewrite meta titles, descriptions, and snippet tags.

Prune or Consolidate (PRUNE_OR_MERGE): Stale pages (content_age_days > 365) with low traffic (impressions_90d < 100) facing continuous decay. Action: 301 redirect to primary category hub or unpublish/410.

Monitor Baseline (MONITOR): Performance metrics remain within expected operating thresholds. Action: No editorial intervention required.

Reason Codes
DEC_IMP_30D: Severe impression velocity loss over the past 30 days.

LOW_CTR_HIGH_POS: Favorable search engine ranking underperforming on user clicks.

STALE_LOW_ENGAGEMENT: Aging asset with sub-threshold traffic and engagement.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 Code: Generating Ranked Queue and Mapping Reason Codes
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# 1. Setup environment and load data
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 2. Leakage-Free Target and Feature Selection
if "target" not in df.columns:
    df["target"] = (df["trend_direction"] == "down").astype(int)

group_col = "client_id" if "client_id" in df.columns else "domain_hash"
leakage_cols = ["target", "id", "trend_direction", "trend_pct"]
features = [c for c in df.select_dtypes(include=[np.number]).columns if c not in leakage_cols]

# Grouped Split: Fit on 80% of clients, evaluate on 20% unseen clients
np.random.seed(42)
unique_clients = df[group_col].unique()
train_clients = np.random.choice(unique_clients, size=int(0.8 * len(unique_clients)), replace=False)

train_df = df[df[group_col].isin(train_clients)].dropna(subset=features + ["target"])
test_df = df[~df[group_col].isin(train_clients)].dropna(subset=features + ["target"]).copy()

model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(train_df[features], train_df["target"])

test_df["decline_prob"] = model.predict_proba(test_df[features])[:, 1]
test_df["pred"] = (test_df["decline_prob"] >= 0.5).astype(int)

# 3. Action Mapping Logic
def assign_action(row):
    if row["pred"] == 1 and row["impressions_90d"] >= 500:
        return "REWRITE_NOW", "DEC_IMP_30D", 1
    elif row["avg_position"] <= 15 and row["ctr"] < 0.02:
        return "CTR_OPTIMIZE", "LOW_CTR_HIGH_POS", 2
    elif row["pred"] == 1 and row["content_age_days"] > 365 and row["impressions_90d"] < 100:
        return "PRUNE_OR_MERGE", "STALE_LOW_ENGAGEMENT", 3
    else:
        return "MONITOR", "STABLE_PERFORMANCE", 4

mapped = test_df.apply(assign_action, axis=1)
test_df["recommended_action"] = [m[0] for m in mapped]
test_df["reason_code"] = [m[1] for m in mapped]
test_df["priority_rank"] = [m[2] for m in mapped]

queue_df = test_df.sort_values(by=["priority_rank", "decline_prob"], ascending=[True, False])

print("=== Ranked Action Queue Breakdown ===")
print(queue_df["recommended_action"].value_counts().to_string())
print("\n=== Sample Priority Queue (Top 5 Rows) ===")
print(queue_df[["decline_prob", "recommended_action", "reason_code"]].head(5).to_string(index=False))

=== Ranked Action Queue Breakdown ===
recommended_action
REWRITE_NOW       3924
MONITOR           1480
CTR_OPTIMIZE       380
PRUNE_OR_MERGE       3

=== Sample Priority Queue (Top 5 Rows) ===
 decline_prob recommended_action reason_code
     0.839404        REWRITE_NOW DEC_IMP_30D
     0.836942        REWRITE_NOW DEC_IMP_30D
     0.819147        REWRITE_NOW DEC_IMP_30D
     0.812166        REWRITE_NOW DEC_IMP_30D
     0.806750        REWRITE_NOW DEC_IMP_30D


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Scope & Operating Limits
Primary Intended Use: Designed strictly as an editorial decision-support system to help SEO leads and content managers prioritize human review workflows.

Operational Boundaries:

Non-Causal Output: Probabilities reflect correlation with historical decay indicators, not causal drivers (e.g., cannot isolate technical outages or tracking script errors from content quality decline).

Niche Seasonality Sensitivity: High-seasonality sites (e.g., holiday or event traffic) may trigger false-positive decline alerts during predictable off-peak months.

No Brand Context: The model processes numeric traffic metrics and lacks context on strategic brand messaging or corporate positioning.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 Code: Cost vs. Protected Value Model
cost_lookup = {"REWRITE_NOW": 150, "CTR_OPTIMIZE": 30, "PRUNE_OR_MERGE": 20, "MONITOR": 0}
queue_df["estimated_cost_usd"] = queue_df["recommended_action"].map(cost_lookup)

# Estimated Protected Value: High-traffic pages saved from complete decay
queue_df["estimated_value_saved_usd"] = np.where(
    queue_df["recommended_action"] == "REWRITE_NOW",
    queue_df["impressions_90d"] * 0.03 * 1.5, # Assumed 3% CTR * $1.50 CPC equivalent
    0
)

roi_summary = queue_df.groupby("recommended_action").agg(
    page_count=("decline_prob", "count"),
    total_cost_usd=("estimated_cost_usd", "sum"),
    total_value_saved_usd=("estimated_value_saved_usd", "sum")
)

print("=== Estimated Playbook Cost & ROI Projection ===")
print(roi_summary.round(2).to_string())

=== Estimated Playbook Cost & ROI Projection ===
                    page_count  total_cost_usd  total_value_saved_usd
recommended_action                                                   
CTR_OPTIMIZE               380           11400                    0.0
MONITOR                   1480               0                    0.0
PRUNE_OR_MERGE               3              60                    0.0
REWRITE_NOW               3924          588600              1342094.8


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Verification Guidelines
Editors must inspect candidates against the following checks before taking action:

Core Business / Converting Landing Pages: Pages generating direct revenue must undergo cross-departmental approval before altering content or structure.

URL Restructuring Check: Confirm whether recent traffic changes reflect site migrations or updated permalink structures.

Editorial Quality: Verify that suggested updates match brand voice and maintain domain authority.

🚫 The No-Go List (Never Automate)
Automated Deletions / 410 Removal: Never allow an automated script to unpublish or delete pages without human review.

Bulk Unchecked 301 Redirect Loops: Never automate bulk redirects without validating destination canonical targets.

Direct Copy Overwrites: Never push machine-generated content directly to live production without editorial oversight.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 Code: Flagging High-Risk Candidates for Mandatory Human Approval
test_df["requires_human_approval"] = np.where(
    (test_df["recommended_action"] == "PRUNE_OR_MERGE") | (test_df["impressions_90d"] > 2000),
    True,
    False
)

flagged_count = test_df["requires_human_approval"].sum()
print(f"Human Audit Trigger: {flagged_count} of {len(test_df)} pages ({flagged_count / len(test_df):.1%}) flagged for mandatory manual approval.")

Human Audit Trigger: 3203 of 5787 pages (55.3%) flagged for mandatory manual approval.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

System Monitoring & Model StalenessRecommendations are considered stale and require retraining if any of these conditions are met:Data Drift Threshold: Feature distribution median values (e.g., ctr or impressions_last_30d) shift by $> 15\%$ month-over-month.Precision Decay: Precision on human-audited sample queues drops below $0.60$ across two consecutive evaluation periods.Algorithm Update Pause: Pause automated queueing for $30$ days following confirmed major Google Core Updates to prevent training on uncalibrated ranking volatility.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 Code: Measuring Feature Shift & Drift
baseline_ctr = train_df["ctr"].median()
current_ctr = test_df["ctr"].median()
drift_pct = abs(current_ctr - baseline_ctr) / baseline_ctr

print(f"Baseline Median CTR: {baseline_ctr:.4f}")
print(f"Evaluation Median CTR: {current_ctr:.4f}")
print(f"Drift Ratio: {drift_pct:.2%}")

if drift_pct > 0.15:
    print("⚠️ ALERT: Drift exceeds 15% threshold. Model re-calibration required.")
else:
    print("✅ STATUS: Operational metrics within stable bounds.")

Baseline Median CTR: 0.0500
Evaluation Median CTR: 0.1500
Drift Ratio: 200.00%
⚠️ ALERT: Drift exceeds 15% threshold. Model re-calibration required.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exporting Artifacts
Exporting the generated queue CSV and metric receipts into work/outputs/ and key charts into work/figures/ for use in the final paper.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5 Code: Export Queue CSV, Metrics JSON, and Reusable Figures
import os
import json
import matplotlib.pyplot as plt
import numpy as np

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Safety check: ensure column exists on queue_df before exporting
if "requires_human_approval" not in queue_df.columns:
    queue_df["requires_human_approval"] = np.where(
        (queue_df["recommended_action"] == "PRUNE_OR_MERGE") | (queue_df["impressions_90d"] > 2000),
        True,
        False
    )

# 1. Export Action Queue CSV
export_cols = ["decline_prob", "recommended_action", "reason_code", "priority_rank", "requires_human_approval", "estimated_cost_usd"]
queue_df[export_cols].to_csv("work/outputs/ranked_action_queue.csv", index=True)
print("Saved queue to 'work/outputs/ranked_action_queue.csv'")

# 2. Export Metrics JSON Receipts
flagged_count = int(queue_df["requires_human_approval"].sum())
metrics_payload = {
    "total_evaluated": int(len(queue_df)),
    "rewrite_now": int((queue_df["recommended_action"] == "REWRITE_NOW").sum()),
    "ctr_optimize": int((queue_df["recommended_action"] == "CTR_OPTIMIZE").sum()),
    "prune_or_merge": int((queue_df["recommended_action"] == "PRUNE_OR_MERGE").sum()),
    "monitor": int((queue_df["recommended_action"] == "MONITOR").sum()),
    "estimated_cost_usd": float(queue_df["estimated_cost_usd"].sum()),
    "human_review_required": flagged_count
}

with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics_payload, f, indent=4)
print("Saved metrics to 'work/outputs/playbook_metrics.json'")

# 3. Save Archetype Distribution Chart
plt.figure(figsize=(7, 4))
queue_df["recommended_action"].value_counts().plot(kind="bar", color="#1f77b4", edgecolor="black")
plt.title("Action Archetype Distribution")
plt.xlabel("Recommended Action")
plt.ylabel("Page Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("work/figures/action_archetype_distribution.png", dpi=300)
plt.close()
print("Saved figure to 'work/figures/action_archetype_distribution.png'")

Saved queue to 'work/outputs/ranked_action_queue.csv'
Saved metrics to 'work/outputs/playbook_metrics.json'
Saved figure to 'work/figures/action_archetype_distribution.png'


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.